In [ ]:
# Setup
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from PIL import Image, ImageDraw, ImageFilter
import os, random, time, math

plt.style.use('dark_background')
sns.set_theme(style='darkgrid')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Hyperparameters
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 20
NUM_CLASSES = 11
NUM_JUNK_TRAIN = 18000
NUM_JUNK_TEST = 3000
MODEL_PATH = 'mnist_cnn.pth'

In [ ]:
# Junk dataset — 8 pattern types with data augmentation
class JunkDataset(Dataset):
    PATTERN_TYPES = ['random_strokes', 'noise', 'grid', 'blob', 'zigzag', 'crosshatch', 'dots', 'curves']

    def __init__(self, size, transform=None):
        self.size = size
        self.transform = transform
        # Extra augmentation specifically for junk images
        self.junk_augment = transforms.Compose([
            transforms.RandomRotation(30),
            transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), scale=(0.7, 1.3), shear=15),
            transforms.RandomPerspective(distortion_scale=0.3, p=0.5),
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.RandomVerticalFlip(p=0.3),
        ])

    def __len__(self):
        return self.size

    def _draw_random_strokes(self, draw):
        for _ in range(random.randint(2, 7)):
            x1, y1 = random.randint(0, 24), random.randint(0, 24)
            x2, y2 = random.randint(x1, 27), random.randint(y1, 27)
            draw.line([(x1, y1), (x2, y2)], fill=random.randint(120, 255), width=random.randint(1, 4))

    def _draw_noise(self, img_array):
        mask = np.random.random((28, 28)) < random.uniform(0.05, 0.3)
        img_array[mask] = np.random.randint(100, 256, size=mask.sum())
        return img_array

    def _draw_grid(self, draw):
        spacing = random.randint(3, 8)
        color, width = random.randint(100, 220), random.randint(1, 2)
        for x in range(0, 28, spacing):
            draw.line([(x, 0), (x, 27)], fill=color, width=width)
        for y in range(0, 28, spacing):
            draw.line([(0, y), (27, y)], fill=color, width=width)

    def _draw_blob(self, draw):
        for _ in range(random.randint(1, 3)):
            cx, cy = random.randint(5, 22), random.randint(5, 22)
            rx, ry = random.randint(3, 10), random.randint(3, 10)
            draw.ellipse([cx-rx, cy-ry, cx+rx, cy+ry], fill=random.randint(130, 255))

    def _draw_zigzag(self, draw):
        points, x = [], random.randint(0, 5)
        for _ in range(random.randint(4, 8)):
            points.append((x, random.randint(0, 27)))
            x = min(27, x + random.randint(2, 6))
        if len(points) >= 2:
            draw.line(points, fill=random.randint(120, 255), width=random.randint(1, 3))

    def _draw_crosshatch(self, draw):
        color, width = random.randint(100, 240), random.randint(1, 3)
        for _ in range(random.randint(3, 6)):
            x1, y1 = random.randint(0, 27), random.randint(0, 27)
            angle = random.uniform(0, 2 * math.pi)
            length = random.randint(8, 25)
            x2 = max(0, min(27, int(x1 + length * math.cos(angle))))
            y2 = max(0, min(27, int(y1 + length * math.sin(angle))))
            draw.line([(x1, y1), (x2, y2)], fill=color, width=width)

    def _draw_dots(self, draw):
        for _ in range(random.randint(8, 30)):
            x, y, r = random.randint(0, 27), random.randint(0, 27), random.randint(0, 2)
            draw.ellipse([x-r, y-r, x+r, y+r], fill=random.randint(120, 255))

    def _draw_curves(self, draw):
        for _ in range(random.randint(2, 5)):
            x1, y1 = random.randint(0, 18), random.randint(0, 18)
            x2, y2 = random.randint(x1+4, 28), random.randint(y1+4, 28)
            start = random.randint(0, 360)
            draw.arc([x1, y1, x2, y2], start=start, end=start+random.randint(60, 300),
                     fill=random.randint(120, 255), width=random.randint(1, 3))

    def __getitem__(self, idx):
        img = Image.new('L', (28, 28), color=0)
        draw = ImageDraw.Draw(img)
        pattern = random.choice(self.PATTERN_TYPES)

        if pattern == 'noise':
            arr = np.array(img)
            arr = self._draw_noise(arr)
            img = Image.fromarray(arr)
        else:
            getattr(self, f'_draw_{pattern}')(draw)

        # Gaussian blur for variety
        if random.random() > 0.4:
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 1.5)))

        # Junk-specific augmentation (rotation, flip, shear, perspective)
        img = self.junk_augment(img)

        if self.transform:
            img = self.transform(img)

        return img, 10


print('JunkDataset ready (8 patterns + augmentation)')

In [ ]:
# Data loading with augmentation
train_transform = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=train_transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=test_transform)
junk_train = JunkDataset(NUM_JUNK_TRAIN, transform=train_transform)
junk_test = JunkDataset(NUM_JUNK_TEST, transform=test_transform)

train_dataset = ConcatDataset([mnist_train, junk_train])
test_dataset = ConcatDataset([mnist_test, junk_test])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train: {len(train_dataset)} (MNIST: {len(mnist_train)}, Junk: {len(junk_train)})')
print(f'Test:  {len(test_dataset)} (MNIST: {len(mnist_test)}, Junk: {len(junk_test)})')

In [ ]:
# Visualize samples
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('MNIST Digits (augmented)', fontsize=16, color='white')
for i, ax in enumerate(axes.flat):
    image, label = mnist_train[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f'{label}', fontsize=10, color='#00ff88')
    ax.axis('off')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Junk Samples (augmented)', fontsize=16, color='white')
for i, ax in enumerate(axes.flat):
    image, label = junk_train[i]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title('junk', fontsize=10, color='#ff6090')
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Model — 3-block VGG-style CNN
class MNISTNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(MNISTNet, self).__init__()
        # Block 1: 1 -> 32
        self.conv1a = nn.Conv2d(1, 32, 3, padding=1)
        self.bn1a = nn.BatchNorm2d(32)
        self.conv1b = nn.Conv2d(32, 32, 3, padding=1)
        self.bn1b = nn.BatchNorm2d(32)
        # Block 2: 32 -> 64
        self.conv2a = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2a = nn.BatchNorm2d(64)
        self.conv2b = nn.Conv2d(64, 64, 3, padding=1)
        self.bn2b = nn.BatchNorm2d(64)
        # Block 3: 64 -> 128
        self.conv3a = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3a = nn.BatchNorm2d(128)
        self.conv3b = nn.Conv2d(128, 128, 3, padding=1)
        self.bn3b = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout_conv = nn.Dropout2d(0.25)
        self.dropout_fc = nn.Dropout(0.5)
        # 28->14->7->3 => 128*3*3 = 1152
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.bn_fc = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = self.pool(F.relu(self.bn1b(self.conv1b(x))))
        x = self.dropout_conv(x)

        x = F.relu(self.bn2a(self.conv2a(x)))
        x = self.pool(F.relu(self.bn2b(self.conv2b(x))))
        x = self.dropout_conv(x)

        x = F.relu(self.bn3a(self.conv3a(x)))
        x = self.pool(F.relu(self.bn3b(self.conv3b(x))))
        x = self.dropout_conv(x)

        x = x.view(x.size(0), -1)
        x = F.relu(self.bn_fc(self.fc1(x)))
        x = self.dropout_fc(x)
        return self.fc2(x)


model = MNISTNet().to(device)
print(model)
print(f'\nParams: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Training setup
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

In [ ]:
# Training loop
train_losses, train_accuracies = [], []
test_losses, test_accuracies = [], []
best_acc = 0.0

for epoch in range(NUM_EPOCHS):
    start = time.time()

    # Train
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100. * correct / total
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # Eval
    model.eval()
    test_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            test_loss += criterion(outputs, labels).item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_loss /= len(test_loader)
    test_acc = 100. * correct / total
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)

    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    marker = ''
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), MODEL_PATH)
        marker = ' *'

    print(f'[{epoch+1}/{NUM_EPOCHS}] ({time.time()-start:.1f}s) '
          f'train: {train_loss:.4f} / {train_acc:.2f}%  '
          f'test: {test_loss:.4f} / {test_acc:.2f}%  '
          f'lr: {lr:.6f}{marker}')

print(f'\nBest test accuracy: {best_acc:.2f}%')

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
e = range(1, NUM_EPOCHS + 1)

ax1.plot(e, train_losses, 'o-', color='#00ff88', label='Train', lw=2)
ax1.plot(e, test_losses, 's-', color='#ff6090', label='Test', lw=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(e, train_accuracies, 'o-', color='#00e5ff', label='Train', lw=2)
ax2.plot(e, test_accuracies, 's-', color='#ffd740', label='Test', lw=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
labels = [str(i) for i in range(10)] + ['Junk']

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu', ax=ax,
            xticklabels=labels, yticklabels=labels, linewidths=0.5, linecolor='gray')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix (11 classes)')
plt.tight_layout(); plt.show()

# Per-class accuracy
for i in range(NUM_CLASSES):
    acc = cm[i, i] / cm[i].sum() * 100
    name = f'Digit {i}' if i < 10 else 'Junk   '
    print(f'  {name}: {acc:.1f}%')

In [ ]:
# Sample predictions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
indices = np.random.choice(len(test_dataset), 10, replace=False)

for i, ax in enumerate(axes.flat):
    image, true_label = test_dataset[indices[i]]
    with torch.no_grad():
        probs = F.softmax(model(image.unsqueeze(0).to(device)), dim=1).cpu().numpy()[0]
    pred = probs.argmax()
    ax.imshow(image.squeeze(), cmap='gray')
    c = '#00ff88' if pred == true_label else '#ff4444'
    ax.set_title(f'P:{pred if pred<10 else "J"} ({probs[pred]*100:.1f}%) T:{true_label if true_label<10 else "J"}',
                 fontsize=10, color=c)
    ax.axis('off')

plt.suptitle('Predictions (Green=Correct, Red=Wrong)', fontsize=14, color='white')
plt.tight_layout(); plt.show()

In [ ]:
# Download model
from google.colab import files

print(f'Size: {os.path.getsize(MODEL_PATH)/1024:.1f} KB')
print(f'Best accuracy: {best_acc:.2f}%')
print('Place downloaded file in: model/mnist_cnn.pth')

files.download(MODEL_PATH)